# **Stack seg images**

#### In this notebook, we will subtract background from ome-tiff images

In [ ]:
# Import packages
import sys
from pathlib import Path
src_path = str(Path.cwd().parent)
if src_path not in sys.path:
    sys.path.append(src_path)


%matplotlib notebook
%matplotlib inline


import bioio_ome_tiff
import bioio_tifffile



## Import images

##### **Add path for a folder of images to be analyzed.**
##### Example: exp_dir = "/Users/kwu2/Documents/Experiments/mRcaax594_MBP647"


In [ ]:
input_dirpath = Path(input())

## Separate segmentations into separate folders

In [ ]:
proc_dirpath = utils.get_proc_dirpath(input_dirpath)
labkit_seg_dirpath = proc_dirpath / dn.labbkit_seg_dirname
labkit_seg_dirpath.mkdir(exist_ok=True)

seg_dirpath = labkit_seg_dirpath / 'cmpcaaxseg'
seg_dirpath.mkdir(exist_ok=True)

caaxoutlineseg_dirpath = labkit_seg_dirpath / 'caaxoutlineseg'
caaxoutlineseg_dirpath.mkdir(exist_ok=True)

for imgpath in input_dirpath.glob('*.ome.tif'):
    if '_seg' in imgpath.name:
        imgpath.rename(seg_dirpath / imgpath.name)
    if '_caaxoutlineseg' in imgpath.name:
        imgpath.rename(caaxoutlineseg_dirpath / imgpath.name)

In [ ]:
# Input directory containing image to add segmentations to
img_dirpath = Path(input())

In [ ]:
img_outlineseg_dirpath = labkit_seg_dirpath / (img_dirpath.name + '_outlineseg')
img_outlineseg_dirpath.mkdir(parents=True, exist_ok=True)

img_seg_init_dirpath = labkit_seg_dirpath / (img_dirpath.name + '_seg_init')
init_fig_dirpath = img_seg_init_dirpath / dn.figs_dirname
init_fig_dirpath.mkdir(parents=True, exist_ok=True)

img_seg_dirpath = labkit_seg_dirpath / (img_dirpath.name + '_seg')
fig_dirpath = img_seg_dirpath / dn.figs_dirname
fig_dirpath.mkdir(parents=True, exist_ok=True)

In [ ]:
# Add caaxoutline segmentations to images

segpaths = [p for p in caaxoutlineseg_dirpath.glob('*.ome.tif')]
segpaths.sort()

for segpath in tqdm(segpaths):

    basename = segpath.name.split('_ROI')[0] + '.ome.tif'
    imgpath = img_dirpath / basename

    stackname = segpath.name.replace('_caaxoutlineseg', '')
    stackpath = img_outlineseg_dirpath / stackname
    if not stackpath.is_file():    
        if imgpath.is_file():
            
            orig_img_file = BioImage(imgpath, reader=bioio_ome_tiff.Reader)
            orig_img = orig_img_file.data
            size_t, _, size_z, size_y, size_x = orig_img.shape

            outlineseg = BioImage(segpath, reader=bioio_tifffile.Reader).data
            outlineseg = outlineseg.reshape(size_t, 1, size_z, size_y, size_x)
            
            outlineseg = ndi.binary_dilation(outlineseg, axes=(3, 4))
            outlineseg = (am.fill_holes_and_remove_uncon_areas(outlineseg, num_areas=3) > 0).astype('bool')

            stack = np.concatenate([orig_img, outlineseg], axis=1)
            stack = stack.astype(orig_img_file.dtype)
            ome_metadata = utils.construct_ome_metadata(stack, orig_img_file)
            OmeTiffWriter.save(stack, stackpath, ome_xml=ome_metadata)

        else:
            print(f'{basename} not found.')
            

## Inspect caaxcell segmentations before next step (manually correct if needed)

In [ ]:
def refine_cmp_cell_seg(seg_caax, seg_cell, num_unconn_areas=3, cell_hole_thresh=3, caax_hole_thresh=3, min_caax_size=3, clean_border_dil=1):
    
    # clean up cell segmentation
    seg_cell = (am.remove_unconnected_areas(seg_cell, num_areas=num_unconn_areas) > 0).astype('bool')
    seg_cell = am.remove_small_holes(seg_cell, area_threshold=cell_hole_thresh)
    
    # mask caax channel with cell channel
    seg_caax = am.mask_img(seg_caax, seg_cell)
    seg_caax = am.remove_small_holes(seg_caax, area_threshold=caax_hole_thresh)
    seg_caax = am.remove_small_objects(seg_caax, min_size=min_caax_size)

    # get cmp areas from newly calculated cellmask and caax
    seg_cmp = (seg_cell & ~seg_caax)
    seg_cmp = am.remove_cmp_near_cellborders(seg_cell, seg_cmp, iter_dil=clean_border_dil)

    # recalculate seg_cell with edited cmp
    seg_cell = (seg_caax | seg_cmp)
    seg_cell = (am.remove_unconnected_areas(seg_cell, num_areas=num_unconn_areas) > 0).astype('bool')

    # re-mask caax seg with edited cell seg
    seg_caax = (seg_caax & seg_cell)

    return seg_caax, seg_cell

In [ ]:
# Add caaxcell segmentations to images

for segpath in tqdm(list(seg_dirpath.glob('*.ome.tif'))):
    basename = segpath.name.split('_ROI')[0] + '.ome.tif'
    imgpath = img_dirpath / basename

    stackname = segpath.name.replace('_seg', '')
    stackpath = img_seg_init_dirpath / stackname
    if not stackpath.is_file():    
        if imgpath.is_file():
            
            orig_img_file = BioImage(imgpath, reader=bioio_ome_tiff.Reader)
            orig_img = orig_img_file.data
            size_t, _, size_z, size_y, size_x = orig_img.shape
    
            seg = BioImage(segpath, reader=bioio_tifffile.Reader).data
            seg = seg.reshape(size_t, 1, size_z, size_y, size_x)
            
            seg_caax = (seg == 1).astype('bool')
            seg_cmp = (seg == 2).astype('bool')
            
            # if separate outline segmentation exists for this image, use it to mask cmp segmentation
            outlinesegpath = img_outlineseg_dirpath / stackname
            if outlinesegpath.is_file():
                outlineseg = BioImage(outlinesegpath, reader=bioio_tifffile.Reader).data
                outlineseg = (outlineseg[:, -1, np.newaxis, :, :, :] > 0)
            else:
            # if no separate file exists, mask with caax seg
                outlineseg = ndi.binary_dilation(seg_caax, axes=(3, 4), iterations=2)
                outlineseg = (am.fill_holes_and_remove_uncon_areas(outlineseg, num_areas=3) > 0).astype('bool')

            seg_cmp = np.where(outlineseg==0, 0, seg_cmp)
            seg_caax = np.where(outlineseg==0, 0, seg_caax)
            seg_cell = (seg_cmp | seg_caax)

            seg_caax, seg_cell = refine_cmp_cell_seg(seg_caax, seg_cell, num_unconn_areas=4, cell_hole_thresh=10, caax_hole_thresh=3, min_caax_size=3, clean_border_dil=3)
            seg_cmp = (seg_cell & ~seg_caax)
            
            # save figure for quick visualization
            cmp_fig = vr.create_cmp_fig(seg_cmp, seg_caax)
            OmeTiffWriter.save(cmp_fig.squeeze(), init_fig_dirpath / stackname)   
            
            # save stacked image
            stack = np.concatenate([orig_img, seg_cmp, seg_caax, cmp_fig], axis=1)
            stack = stack.astype(orig_img_file.dtype)
            ome_metadata = utils.construct_ome_metadata(stack, orig_img_file)
            OmeTiffWriter.save(stack, stackpath, ome_xml=ome_metadata)   
            
        else:
            print(f'{basename} not found.')

In [ ]:
def edit_cmp_regions(seg_cmp, seg_caax, bg_hole_thresh=20, min_perc_caax_border=0.9):

    seg_cell = (seg_cmp | seg_caax)
    size_t = seg_cmp.shape[0]
    for t in range(size_t):
        seg_cmp_t = seg_cmp[t, :, :, :, :]
        seg_caax_t = seg_caax[t, :, :, :, :]
        seg_cell_t = seg_cell[t, :, :, :, :]
    
        labels_to_remove = []

        if bg_hole_thresh != None:
            # omit compaction regions containing significant background pixels
            cmp_fill = ndi.binary_fill_holes(seg_cmp_t, axes=(-1, -2))
            cmp_fill_labelled = morphology.label(cmp_fill)
            cmp_labelled = np.where(seg_cmp_t > 0, cmp_fill_labelled, 0)
            cmp_holes_labelled = np.where(seg_cell_t==0, cmp_fill_labelled, 0)
            hole_props = measure.regionprops_table(cmp_holes_labelled.squeeze(), properties=['label', 'area'])
            hole_df = pd.DataFrame(hole_props)
            labels_bg_hole = hole_df.loc[hole_df['area'] > bg_hole_thresh, 'label'].values
            print(f't={t}: compaction regions removed for containing background holes: {len(labels_bg_hole)}')
            labels_to_remove.extend(labels_bg_hole)

        if min_perc_caax_border != None:
            # omit compaction regions that are not mostly bordered by caax+ areas
            cmp_expanded = segmentation.expand_labels(cmp_labelled)
            border_labelled = np.where(seg_cmp_t > 0, 0, cmp_expanded)
            border_props = measure.regionprops_table(border_labelled.squeeze(),
                                  properties=['label', 'area'])
            caax_border_labelled = np.where(seg_caax_t > 0, border_labelled, 0)
            caax_border_props = measure.regionprops_table(caax_border_labelled.squeeze(),
                              properties=['label', 'area'])
            border_df = pd.DataFrame(border_props)
            border_df = border_df.rename(columns={'area': 'border area'})
            caax_border_df = pd.DataFrame(caax_border_props)
            caax_border_df = caax_border_df.rename(columns={'area': 'caax border area'})
            df = border_df.merge(caax_border_df, how='left', on='label')
            df['perc_caax_border'] = df['caax border area'] / df['border area']
            labels_bg_border = df.loc[df['perc_caax_border'] < min_perc_caax_border, 'label'].values
            print(f't={t}: compaction regions removed for bordering background pixels: {len(labels_bg_border)}')
            labels_to_remove.extend(labels_bg_border)
    
        # remove compaction areas
        labels_to_remove = np.unique(labels_to_remove)
        for label in labels_to_remove:     
            seg_cmp_t = np.where(cmp_labelled==label, 0, seg_cmp_t)
    
        seg_cmp[t, :, :, :, :] = seg_cmp_t
        
    return seg_cmp

In [ ]:
mask_dirpath = Path(input())

In [ ]:
metadata_imgpath = Path(input())

# Before next step, draw polygon ROIs and check segmentations (correct if necessary)

In [ ]:
# extract metadata (mostly pixel sizes) from an image that still has metadata attached
img_file = BioImage(metadata_imgpath, reader=bioio_ome_tiff.Reader)

# Mask images and clean segmentations
seg_cmpch = -3
seg_caaxch = -2
segch = -1

for imgpath in tqdm(list(img_seg_init_dirpath.glob('*.ome.tif'))):

    
    savepath = img_seg_dirpath / imgpath.name
    if not savepath.is_file():    
        print(savepath.name)
        maskpath = mask_dirpath / (imgpath.name.replace('.ome.tif', '.png'))
        if maskpath.is_file():
            
            img = BioImage(imgpath, reader=bioio_tifffile.Reader).data
            seg = img[:, segch, np.newaxis, :, :, :]

            # apply mask
            polygonmask = (np.array(Image.open(maskpath)) > 0).astype('int')
            seg = np.where(polygonmask==0, 0, seg)

            # edit segmentations
            seg_cell = (seg > 0)
            seg_cell = am.remove_small_holes(seg_cell, area_threshold=5)
            seg_caax = ((seg==100) & seg_cell)
            seg_caax = am.remove_small_holes(seg_caax, area_threshold=3)
            seg_caax = am.remove_small_objects(seg_caax, min_size=3)
            seg_cmp = (seg_cell & ~seg_caax)
            seg_cmp = edit_cmp_regions(seg_cmp, seg_caax, bg_hole_thresh=10, min_perc_caax_border=0.95)
            seg_cell = (seg_caax | seg_cmp)
            seg_cell = (am.remove_unconnected_areas(seg_cell, num_areas=1) > 0).astype('int')
            seg_caax = (seg_caax & seg_cell)
            seg_cmp = (seg_cell & ~seg_caax)
            
            # save figure for quick visualization
            cmp_fig = vr.create_cmp_fig(seg_cmp, seg_caax)
            OmeTiffWriter.save(cmp_fig, fig_dirpath / imgpath.name)
            
            # save stacked image
            img[:, [seg_cmpch, seg_caaxch, segch], :, :, :] = np.concatenate([seg_cmp, seg_caax, cmp_fig], axis=1)
            img = img.astype(img_file.dtype)
            ome_metadata = utils.construct_ome_metadata(img, img_file)
            OmeTiffWriter.save(img, savepath, ome_xml=ome_metadata)      
            
        else:
            print(f'{maskpath.name} not found.')

In [ ]:
# Resave images as ome-tiffs after additional manual editing

# extract metadata (mostly pixel sizes) from an image that still has metadata attached
img_file = BioImage(metadata_imgpath, reader=bioio_ome_tiff.Reader)

# Mask images and clean segmentations
seg_cmpch = -3
seg_caaxch = -2
segch = -1

for imgpath in tqdm(list(img_seg_dirpath.glob('*.ome.tif'))):

    
    savepath = img_seg_dirpath / imgpath.name
    maskpath = mask_dirpath / (imgpath.name.replace('.ome.tif', '.png'))
    if maskpath.is_file():
            
        img = BioImage(imgpath, reader=bioio_tifffile.Reader).data
        seg = img[:, segch, np.newaxis, :, :, :]

        # resave segmentations
        seg_cell = (seg > 0)
        seg_caax = ((seg==100) & seg_cell)
        seg_cmp = (seg_cell & ~seg_caax)
        seg_cell = (am.remove_unconnected_areas(seg_cell, num_areas=1) > 0).astype('int')
        seg_caax = (seg_caax & seg_cell)
        seg_cmp = (seg_cell & ~seg_caax)
        
        # save figure for quick visualization
        cmp_fig = vr.create_cmp_fig(seg_cmp, seg_caax)
        OmeTiffWriter.save(cmp_fig, fig_dirpath / imgpath.name)
        
        # save stacked image
        img[:, [seg_cmpch, seg_caaxch, segch], :, :, :] = np.concatenate([seg_cmp, seg_caax, cmp_fig], axis=1)
        img = img.astype(img_file.dtype)
        ome_metadata = utils.construct_ome_metadata(img, img_file)
        OmeTiffWriter.save(img, savepath, ome_xml=ome_metadata)      
        
    else:
        print(f'{maskpath.name} not found.')

# Manually edit segmentations
#### Recommended: after manually editing segmentations, create a copy of the folder before moving on to the next step

In [ ]:
# extract metadata (mostly pixel sizes) from an image that still has metadata attached
img_file = BioImage(metadata_imgpath, reader=bioio_ome_tiff.Reader)

# Mask images and clean segmentations
seg_cmpch = -3
seg_caaxch = -2
segch = -1

for imgpath in tqdm(list(img_seg_dirpath.glob('*.ome.tif'))):
    
    savepath = img_seg_dirpath / imgpath.name
            
    img = BioImage(imgpath, reader=bioio_tifffile.Reader).data
    seg = img[:, segch, np.newaxis, :, :, :]

    # edit segmentations
    seg_cell = (seg > 0)
    seg_caax = ((seg==100) & seg_cell)
    seg_cmp = (seg_cell & ~seg_caax)
    seg_cmp = edit_cmp_regions(seg_cmp, seg_caax, bg_hole_thresh=10, min_perc_caax_border=0.95)
    seg_cell = (seg_caax | seg_cmp)
    seg_cell = (am.remove_unconnected_areas(seg_cell, num_areas=1) > 0).astype('int')
    seg_caax = (seg_caax & seg_cell)
    seg_cmp = (seg_cell & ~seg_caax)
    
    # save figure for quick visualization
    cmp_fig = vr.create_cmp_fig(seg_cmp, seg_caax)
    OmeTiffWriter.save(cmp_fig, fig_dirpath / imgpath.name)
    
    # save stacked image
    img[:, [seg_cmpch, seg_caaxch, segch], :, :, :] = np.concatenate([seg_cmp, seg_caax, cmp_fig], axis=1)
    img = img.astype(img_file.dtype)
    ome_metadata = utils.construct_ome_metadata(img, img_file)
    OmeTiffWriter.save(img, savepath, ome_xml=ome_metadata)      


In [ ]:
imgpath = Path(input())

In [ ]:
overlap_imgpath = Path(input())

In [ ]:
img_file = BioImage(metadata_imgpath, reader=bioio_ome_tiff.Reader)

img = BioImage(imgpath, reader=bioio_tifffile.Reader).data
overlap_img = BioImage(overlap_imgpath, reader=bioio_tifffile.Reader).data

img_seg = img[:, -3:, :, :, :]
overlap_seg = overlap_img[:, -3:, :, :, :]
img[:, -3:, :, :, :] = np.where(overlap_seg > 0, 0, img_seg)
ome_metadata = utils.construct_ome_metadata(img, img_file)
OmeTiffWriter.save(img, imgpath, ome_xml=ome_metadata)  



In [ ]:
img_seg_dirpath = Path(input())

img_seg_init_dirpath = Path(input())

fig_dirpath = Path(input())

In [ ]:
# extract metadata (mostly pixel sizes) from an image that still has metadata attached
img_file = BioImage(metadata_imgpath, reader=bioio_ome_tiff.Reader)

# Mask images and clean segmentations
seg_cmpch = -3
seg_caaxch = -2
segch = -1

for imgpath in tqdm(list(img_seg_init_dirpath.glob('*.ome.tif'))):
    
    savepath = img_seg_dirpath / imgpath.name
            
    img = BioImage(imgpath, reader=bioio_tifffile.Reader).data
    seg = img[:, segch, np.newaxis, :, :, :]

    # edit segmentations
    seg_cell = (seg > 0)
    seg_caax = ((seg==100) & seg_cell)
    seg_cmp = (seg_cell & ~seg_caax)
    seg_cmp = edit_cmp_regions(seg_cmp, seg_caax, bg_hole_thresh=10, min_perc_caax_border=0.95)
    seg_cell = (seg_caax | seg_cmp)
    seg_cell = (am.remove_unconnected_areas(seg_cell, num_areas=1) > 0).astype('int')
    seg_caax = (seg_caax & seg_cell)
    seg_cmp = (seg_cell & ~seg_caax)
    
    # save figure for quick visualization
    cmp_fig = vr.create_cmp_fig(seg_cmp, seg_caax)
    OmeTiffWriter.save(cmp_fig, fig_dirpath / imgpath.name)
    
    # save stacked image
    img[:, [seg_cmpch, seg_caaxch, segch], :, :, :] = np.concatenate([seg_cmp, seg_caax, cmp_fig], axis=1)
    img = img.astype(img_file.dtype)
    ome_metadata = utils.construct_ome_metadata(img, img_file)
    OmeTiffWriter.save(img, savepath, ome_xml=ome_metadata)      
